# ML-testbed 16 — build shared ontology and provenance

Lesson 15 coordinated bounded work. This lesson builds the durable shared memory those workers need. It explores why organized data affects LLM retrieval, training, and evaluation by loading Factor's ontology kernel, constructing typed knowledge records, comparing retrieval policies, compiling a reproducible context manifest, and filtering a training set by provenance.

No model is called. The lab isolates the **data-selection mechanism** so it can later be compared with identical models and token budgets.

## How to use this lesson

Run from the Factor repository root or `notebooks/` with Python 3.10 or later. No API calls, downloads, model server, or graph database are required.

**Skills:** ontology envelopes, tags, provenance, governed retrieval, immutable context manifests, training-data lineage

**Evidence contract:** the retrieval collection and relevance labels are an exposed teaching example. The comparison explains a mechanism; it is not a measured claim that an ontology improves LLM accuracy on independent tasks.

## The knowledge pipeline

```mermaid
flowchart LR
  R[Raw documents] --> N[Normalize and hash]
  N --> T[Type and tag]
  T --> P[Attach provenance]
  P --> V[Validate]
  V --> K[Versioned knowledge]
  K --> C[Compile authorized context]
  C --> A[LLM or deterministic worker]
  A --> E[Evaluate]
  E -->|governed update| K
```

The context window is a temporary projection. The versioned knowledge system is the durable memory.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((path for path in candidates
             if (path / "docs/ontology/factor-core.schema.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from factor/ or factor/notebooks/")

ontology_schema = json.loads((ROOT / "docs/ontology/factor-core.schema.json").read_text())
example_claim = json.loads((ROOT / "docs/ontology/example-claim.json").read_text())
print("ontology:", example_claim["ontology_version"])
print("object type:", example_claim["object_type"])
print("tags:", ", ".join(example_claim["tags"]))
print("status:", example_claim["verification_status"])

## 1. Inspect the common metadata envelope

This lightweight check explains the fields used below. It is **not** a replacement for a JSON Schema 2020-12 validator.

In [ ]:
required = set(ontology_schema["$defs"]["envelope"]["required"])
missing = required - set(example_claim)
assert not missing, f"missing required fields: {sorted(missing)}"
assert 0 <= example_claim["confidence"] <= 1
assert len(set(example_claim["tags"])) == len(example_claim["tags"])

print(f"required envelope fields present: {len(required)}")
for name in sorted(required):
    print(f"  {name:<24} {example_claim[name]}")

## 2. Build a tiny governed document collection

`relevant` is evaluator truth for this teaching example. A production benchmark would keep it separate from the retrieval code.

In [ ]:
documents = [
    {"id": "source:calibration-note", "text": "Optical drift calibration and motion ambiguity.",
     "type": "Source", "tags": ["domain/pdv", "topic/optical-drift"],
     "status": "accepted", "security": "project", "relevant": True},
    {"id": "artifact:synthetic-drift-run", "text": "Synthetic optical drift reproduces apparent motion.",
     "type": "Artifact", "tags": ["domain/pdv", "topic/optical-drift", "evidence/synthetic"],
     "status": "accepted", "security": "project", "relevant": True},
    {"id": "claim:bounded-inference", "text": "Motion remains ambiguous without a defensible drift bound.",
     "type": "Claim", "tags": ["domain/pdv", "topic/optical-drift"],
     "status": "provisional", "security": "project", "relevant": True},
    {"id": "source:obsolete-note", "text": "Optical drift never affects motion inference.",
     "type": "Source", "tags": ["domain/pdv", "topic/optical-drift"],
     "status": "superseded", "security": "project", "relevant": False},
    {"id": "artifact:generated-summary", "text": "A model says optical drift proves material motion.",
     "type": "Artifact", "tags": ["domain/pdv", "origin/model-generated"],
     "status": "unreviewed", "security": "project", "relevant": False},
    {"id": "source:restricted-shot", "text": "Restricted optical drift and motion record.",
     "type": "Source", "tags": ["domain/pdv", "topic/optical-drift"],
     "status": "accepted", "security": "restricted", "relevant": False},
    {"id": "source:robotics", "text": "Optical flow estimates robot motion.",
     "type": "Source", "tags": ["domain/robotics"],
     "status": "accepted", "security": "public", "relevant": False},
    {"id": "policy:review", "text": "Claims need evidence and review before acceptance.",
     "type": "Policy", "tags": ["governance/claims"],
     "status": "accepted", "security": "project", "relevant": False},
]

for doc in documents:
    doc["sha256"] = hashlib.sha256(doc["text"].encode()).hexdigest()
print(f"collection contains {len(documents)} versionable objects")

## 3. Compare unstructured and governed retrieval

The unstructured baseline sees words only. The governed strategy combines lexical matching with type, tag, verification, and authorization filters.

In [ ]:
query_terms = {"optical", "drift", "motion"}

def lexical_score(doc):
    words = set(doc["text"].lower().replace(".", "").split())
    return len(query_terms & words)

unstructured = sorted(
    (doc for doc in documents if lexical_score(doc) > 0),
    key=lambda doc: (-lexical_score(doc), doc["id"]),
)[:5]

governed = [doc for doc in unstructured
            if "domain/pdv" in doc["tags"]
            and doc["status"] in {"accepted", "provisional"}
            and doc["security"] in {"public", "project"}]

def metrics(results):
    selected = {doc["id"] for doc in results}
    truth = {doc["id"] for doc in documents if doc["relevant"]}
    hits = len(selected & truth)
    return {
        "selected": len(selected),
        "precision": hits / len(selected) if selected else 0,
        "recall": hits / len(truth) if truth else 0,
    }

for label, results in [("unstructured", unstructured), ("governed", governed)]:
    score = metrics(results)
    print(f"{label:<13} selected={score['selected']}  precision={score['precision']:.2f}  recall={score['recall']:.2f}")
    for doc in results:
        print("  ", doc["id"])

This toy result is explanatory, not an empirical LLM-performance claim. A real Factor evaluation must freeze tasks and truth, use identical models and token budgets, and measure citation validity, unsupported claims, accuracy, calibration, latency, and cost.

## 4. Compile an immutable context manifest

The manifest records the exact object IDs, hashes, and selection policy. A changed source creates a different manifest rather than silently altering an old attempt.

In [ ]:
manifest_payload = {
    "ontology_version": "factor-ontology/0.1",
    "objective": "Assess whether optical drift can mimic apparent motion",
    "selection_policy": {
        "domain_tag": "domain/pdv",
        "allowed_status": ["accepted", "provisional"],
        "allowed_security": ["public", "project"],
    },
    "items": [{"id": doc["id"], "sha256": doc["sha256"]} for doc in governed],
}
manifest_hash = hashlib.sha256(
    json.dumps(manifest_payload, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
context_manifest = {
    "id": f"context:{manifest_hash[:16]}",
    "created_at": datetime.now(timezone.utc).isoformat(),
    **manifest_payload,
}

print(context_manifest["id"])
print(json.dumps(context_manifest["items"], indent=2))

## 5. Trace provenance instead of trusting fluent text

Claims and artifacts remain separate. Typed relationships let a reviewer walk from a claim to its evidence and original source.

In [ ]:
relationships = [
    ("claim:bounded-inference", "supportedBy", "artifact:synthetic-drift-run"),
    ("artifact:synthetic-drift-run", "derivedFrom", "source:calibration-note"),
]

def provenance_path(start):
    path, frontier, seen = [], [start], set()
    while frontier:
        current = frontier.pop(0)
        if current in seen:
            continue
        seen.add(current)
        for subject, predicate, obj in relationships:
            if subject == current:
                path.append((subject, predicate, obj))
                frontier.append(obj)
    return path

for subject, predicate, obj in provenance_path("claim:bounded-inference"):
    print(f"{subject} --{predicate}--> {obj}")

## 6. Use provenance to protect a training split

Hashes expose duplicate content. Origin tags and review status let a dataset builder exclude unreviewed model output instead of training on it as if it were independent evidence.

In [ ]:
candidates = documents + [{**documents[0], "id": "source:calibration-note-copy"}]
accepted_examples, excluded, seen_hashes = [], [], set()

for doc in candidates:
    reasons = []
    if doc["sha256"] in seen_hashes:
        reasons.append("duplicate-content")
    if "origin/model-generated" in doc["tags"] and doc["status"] != "accepted":
        reasons.append("unreviewed-model-origin")
    if doc["status"] in {"rejected", "superseded", "unreviewed"}:
        reasons.append(f"status/{doc['status']}")
    if doc["security"] == "restricted":
        reasons.append("not-authorized-for-training")
    if reasons:
        excluded.append((doc["id"], reasons))
    else:
        accepted_examples.append(doc["id"])
        seen_hashes.add(doc["sha256"])

print("accepted training examples:", len(accepted_examples))
print("excluded:")
for object_id, reasons in excluded:
    print(f"  {object_id:<34} {', '.join(reasons)}")

## What this lab establishes

- Types and tags make retrieval policy explicit.
- Provenance distinguishes source evidence from generated interpretation.
- Verification and security filters prevent known-bad or unauthorized context.
- Content hashes help detect duplicates and freeze datasets.
- Context manifests make model-visible inputs replayable.

**Try next:** change a status, tag, or authorization level and observe the manifest ID and retrieval metrics. Then add temporal validity or a contradictory claim without deleting either interpretation.